
# Interactive beam-fit explorer (marjum-2026-07, 91 m beam scan)

Twiddle the sliders, watch the residual respond. Every parameter of the current
fit model is exposed, plus the geometry that has been under dispute.

**Run this cell-by-cell top to bottom, then use the controls at the bottom.**

### What you need locally

| | |
|---|---|
| Python packages | `numpy`, `matplotlib`, `healpy`, `ipywidgets` (and JupyterLab/Notebook) |
| Data | `beam_explorer_cache.npz` (14 MB) -- must sit next to this notebook |

**Nothing else.** No `eigsep_data` install, no correlator HDF5 files, no network.
Everything expensive (226 file reads, the data-space RFI mask, the HFSS PCA) was
precomputed into the cache by `build_explorer_cache.py`. The coupling maths is
reimplemented here in ~25 lines of numpy so the notebook does not depend on the
`eigsep_data` package at all.

Install what is missing with:
```
pip install numpy matplotlib healpy ipywidgets
```

### What the panels show

**Data** | **Model** | **Difference (data - model)**, all in (az, el), binned for
responsiveness. The live **normalized RMS** readout above the plot is the metric
to minimize -- it is the same quantity the automated pipeline optimizes, so a
number you reach by hand is directly comparable to its result.

### Starting point and reference values

Defaults are the surveyed geometry (transmitter 93 m below the antenna) rather
than the pipeline's fitted heading, because the fitted heading was shown to be
~86 deg away from the survey and the fit is insensitive to the difference.
Reference buttons load the other candidates.


In [1]:

import numpy as np
import matplotlib.pyplot as plt
import healpy as hp
import ipywidgets as W
from IPython.display import display

C = np.load("beam_explorer_cache.npz")
AZ = C["az_deg"].astype(float)
EL = C["el_deg"].astype(float)
Y = C["measured_tx"]            # (nchan, nsample) float32
USED = C["used"]                # (nchan, nsample) bool
CHANS = C["channels"]
ARMS = C["arms"]
FREQS = C["freqs_mhz"]
A_HFSS = C["a_hfss"]            # (nchan, ncomp) complex -- HFSS prior per channel
BEAM = C["beam_cart"].astype(np.complex128)   # (ncomp, 3, npix)
NSIDE = int(C["nside"])
NCOMP = BEAM.shape[0]

SURVEYED_DELTA = C["surveyed_delta_enu"]      # antenna -> TX, metres (E, N, U)
HEADING_NEW, ALPHA_NEW = C["heading_new"], float(C["alpha_new"])
HEADING_OLD, ALPHA_OLD = C["heading_old"], float(C["alpha_old"])

print(f"{len(CHANS)} channels, {AZ.size} samples, {NCOMP} PCA components, nside={NSIDE}")
print(f"surveyed antenna->TX offset (E,N,U) = {SURVEYED_DELTA.round(2)} m")
print(f"  = {np.degrees(np.arctan2(-SURVEYED_DELTA[2], np.hypot(*SURVEYED_DELTA[:2]))):.1f}"
      f" deg below horizontal, slant range {np.linalg.norm(SURVEYED_DELTA):.1f} m")


101 channels, 54240 samples, 4 PCA components, nside=32
surveyed antenna->TX offset (E,N,U) = [ -3.67  -5.82 -93.01] m
  = 85.8 deg below horizontal, slant range 93.3 m



## The model

Reimplemented from `eigsep_data.beam_mapping.tx_model.simulate_hfss_coupling`
(production path: azimuth about -z, elevation about +x) so this notebook stands
alone. Identical maths, no package dependency.

**One thing worth knowing before you turn the range knob:** the coupling model is
pure far-field, so only the *direction* to the transmitter affects the pattern.
Changing the transmitter's distance rescales the model but does not reshape it,
and the amplitude fit absorbs that exactly -- so range is degenerate with gain
by construction. The E/N/U sliders are still useful for changing the *direction*.


In [2]:

_TH, _PH = hp.pix2ang(NSIDE, np.arange(hp.nside2npix(NSIDE)))


def rotations(az_deg, el_deg):
    az, el = np.deg2rad(az_deg), np.deg2rad(el_deg)
    ca, sa = np.cos(az), -np.sin(az)        # azimuth about -z (production path)
    ce, se = np.cos(el), np.sin(el)
    R = np.empty((az.size, 3, 3))
    R[:, 0] = np.stack([ca, -sa, np.zeros_like(ca)], axis=1)
    R[:, 1] = np.stack([ce * sa, ce * ca, -se], axis=1)
    R[:, 2] = np.stack([se * sa, se * ca, ce], axis=1)
    return R


def field_top(alpha_deg, arm):
    a = np.deg2rad(alpha_deg + (90.0 if arm else 0.0))
    return np.array([-np.sin(a), np.cos(a), 0.0])


def coupling(az_deg, el_deg, heading, alpha_deg, arm):
    """Complex per-component coupling, shape (ncomp, nsample)."""
    R = rotations(az_deg, el_deg)
    Rt = R.transpose(0, 2, 1)
    rhat = np.einsum("nij,j->ni", Rt, heading)
    th = np.arccos(np.clip(rhat[:, 2], -1, 1))
    ph = np.mod(np.arctan2(rhat[:, 1], rhat[:, 0]), 2 * np.pi)
    px = hp.ang2pix(NSIDE, th, ph)
    w = np.moveaxis(BEAM[:, :, px], 1, -1)              # (ncomp, nsample, 3)
    e = np.einsum("nij,j->ni", Rt, field_top(alpha_deg, arm))
    e = e - np.sum(e * rhat, axis=1, keepdims=True) * rhat
    w = w - np.sum(w * rhat[None], axis=2, keepdims=True) * rhat[None]
    return np.einsum("fni,ni->fn", np.conj(w), e)


def householder(u):
    """Unitary whose first column is proportional to u (fixes the phase gauge)."""
    e0 = np.zeros(u.size, complex); e0[0] = 1.0
    alpha = -np.exp(1j * np.angle(u[0])) if abs(u[0]) > 1e-12 else -1.0
    v = u - alpha * e0
    nv = np.linalg.norm(v)
    if nv < 1e-12:
        return np.eye(u.size, dtype=complex)
    v = v / nv
    return np.eye(u.size, dtype=complex) - 2 * np.outer(v, np.conj(v))


def model_power(ch_index, heading, alpha_deg, arm, shape_re, shape_im,
                gain=None, az_off=0.0, el_off=0.0, apply_cal=False,
                el_sign=-1):
    """Model power per sample. gain=None -> least-squares amplitude (RMS-optimal).

    el_sign = -1 is Christian's hardware convention (+EL tips the boresight
    NORTH); the model's own rotation, about the fixed East shaft, tips it
    SOUTH, so el_sign = +1 reproduces the uncorrected behaviour and every
    number published before 2026-09-17.
    """
    cpl = coupling(AZ + az_off, el_sign * EL + el_off, heading, alpha_deg, arm)
    a = A_HFSS[ch_index]
    g_hfss = np.linalg.norm(a)
    basis = householder(a / max(g_hfss, 1e-30))
    cpl_rot = basis.conj().T @ cpl
    params = np.r_[1.0, np.array(shape_re) + 1j * np.array(shape_im)]
    m = np.abs(np.conj(params) @ cpl_rot) ** 2
    u = chan_used(ch_index, apply_cal)
    d = chan_data(ch_index, apply_cal)
    if gain is None:
        denom = np.sum(m[u] * m[u])
        A = np.sum(d[u] * m[u]) / denom if denom > 0 else 0.0
    else:
        A = gain
    return A * m, A


def normalized_rms(ch_index, model, el_cut=180.0, apply_cal=False):
    u = chan_used(ch_index, apply_cal) & (np.abs(EL) <= el_cut)
    d = chan_data(ch_index, apply_cal)[u]
    r = d - model[u]
    return float(np.sqrt(np.mean(r ** 2)) / max(np.sqrt(np.mean(d ** 2)), 1e-30))


def heading_from_enu(dE, dN, dU):
    v = np.array([dE, dN, dU], float)
    n = np.linalg.norm(v)
    return v / n if n > 0 else np.array([0.0, 0.0, -1.0])


# --- B7 receiver-gain calibration -------------------------------------------
# measured_tx is a channel difference, so with auto = g_rx * (T_sky + T_rx)
# and g_rx smooth over three adjacent 244 kHz channels it carries g_rx as a
# linear multiplicative factor. Dividing it out *should* remove a drift the
# single fit amplitude cannot absorb (x1.61 at 173.83 MHz here). Measured, it
# does the opposite -- see `beam_cal_toggle_checkpoint`.
import os.path as _osp

E_CAL_AVAILABLE = _osp.exists("cal_gain.npz") and "times" in C
CAL_GAIN = None
CAL_ANCHOR_MIN = None
CAL_GAP = (None, None)
if E_CAL_AVAILABLE:
    _cal = np.load("cal_gain.npz", allow_pickle=True)
    TIMES = C["times"].astype(float)
    _st = _cal["sol_times"].astype(float)
    _gc = _cal["gain_cycles"].astype(float)
    CAL_GAIN = np.full((len(CHANS), TIMES.size), np.nan)
    _tok = TIMES > 0
    for _j in range(len(CHANS)):
        _ok = np.isfinite(_gc[:, _j]) & (_gc[:, _j] > 0)
        if _ok.sum() < 2:
            continue
        _s, _g = _st[_ok], _gc[_ok, _j]
        _in = _tok & (TIMES >= _s[0]) & (TIMES <= _s[-1])
        CAL_GAIN[_j, _in] = np.interp(TIMES[_in], _s, _g)
    CAL_ANCHOR_MIN = np.full(TIMES.size, np.nan)
    CAL_ANCHOR_MIN[_tok] = np.min(
        np.abs(TIMES[_tok][:, None] - _st[None, :]), axis=1) / 60.0
    CAL_GAP = (str(_cal["gap_start_utc"]), str(_cal["gap_end_utc"]))
    print(f"B7 gain loaded: {_st.size} cycles; "
          f"unsolved gap {CAL_GAP[0]} -> {CAL_GAP[1]}")
else:
    print("cal_gain.npz not found -- the g_rx toggle will be disabled")


def chan_data(i, apply_cal):
    """Measured quantity: raw accumulator counts, or counts / g_rx."""
    d = Y[i].astype(float)
    if not apply_cal or not E_CAL_AVAILABLE:
        return d
    with np.errstate(invalid="ignore", divide="ignore"):
        return d / CAL_GAIN[i]


def chan_used(i, apply_cal):
    """USED, restricted to samples with a gain solution when cal is on."""
    if not apply_cal or not E_CAL_AVAILABLE:
        return USED[i]
    return USED[i] & np.isfinite(CAL_GAIN[i])


print("model ready")


B7 gain loaded: 66 cycles; unsolved gap 2026-07-17T19:42:14Z -> 2026-07-18T01:24:40Z
model ready



## Binning for display

Scatter-plotting ~25,000 points three times per slider move is too slow to feel
interactive, so samples are binned onto an (az, el) grid once and the bin index
is cached. Each update is then a `bincount`, which is milliseconds. Change
`BIN_DEG` if you want finer or coarser maps.


In [3]:

BIN_DEG = 4.0
AZ_EDGES = np.arange(0.0, 360.0 + BIN_DEG, BIN_DEG)
EL_EDGES = np.arange(-180.0, 180.0 + BIN_DEG, BIN_DEG)
NAZ, NEL = len(AZ_EDGES) - 1, len(EL_EDGES) - 1
_ai = np.clip(np.digitize(AZ, AZ_EDGES) - 1, 0, NAZ - 1)
_ei = np.clip(np.digitize(EL, EL_EDGES) - 1, 0, NEL - 1)
_flat = _ai * NEL + _ei


def grid(values, mask):
    s = np.bincount(_flat[mask], weights=values[mask], minlength=NAZ * NEL)
    n = np.bincount(_flat[mask], minlength=NAZ * NEL)
    out = np.full(NAZ * NEL, np.nan)
    ok = n > 0
    out[ok] = s[ok] / n[ok]
    return out.reshape(NAZ, NEL).T


print(f"display grid: {NAZ} az x {NEL} el bins of {BIN_DEG} deg")


display grid: 90 az x 90 el bins of 4.0 deg



## Controls

**Pure HFSS is the default.** The `pure HFSS (lock shape)` checkbox starts
**on**, which holds `shape1/2/3` at zero and greys them out. In that mode the
model is the unperturbed physical HFSS prediction and the only live knobs are
the physical ones: amplitude, TX direction (E/N/U), and `alpha`. The shape
terms are extra non-physical freedom on top of the prior, not one of this
notebook's intended comparison knobs, so they are locked unless you clear the
box deliberately.

For scale, on the corrected cache the pure-HFSS model reaches a median
normalized RMS of **0.518** across the 101 channels with only the amplitude
free; letting the three shape terms float as well buys **0.469**. So the
non-physical freedom is worth ~0.05 in the median -- worth knowing before you
decide the prior is inadequate.

**`apply g_rx cal (B7)` starts OFF, and that is a finding, not a default.**
The checkbox divides `measured_tx` by rf-calibrator's per-cycle receiver gain
$g_{\rm rx}(\nu,t)$ (`abscal/trx_phaseC.npz`, 66 cycles) interpolated onto the
sample times, and drops samples with no solution. Since `measured_tx` is a
channel *difference* it should carry $g_{\rm rx}$ as a linear factor, so
dividing it out ought to remove a real drift — $\times 1.61$ at 173.83 MHz
across this window, which the single fit amplitude cannot absorb.

**It makes the fit worse.** ch712 pure-HFSS goes 0.391 → 0.486; the fleet
median goes 0.518 → 0.587; 69 of 101 channels degrade. `beam_cal_toggle_checkpoint.ipynb` has the three controls that establish
this is not a bug in the correction. Leave it off for fitting; turn it on to look at the
anomaly.

**Tips for fitting by hand**
- Leave **auto-fit amplitude** on while you shape the model; it keeps the
  amplitude RMS-optimal so you are only judging shape. Turn it off to set gain
  by hand.
- Watch **normalized RMS** (lower is better). `1.0` means the model explains
  nothing; the automated pipeline reaches ~0.90 per-sample on most channels, and
  ~0.50 on the best ones.
- The **|el| cut** restricts *both* the plot and the RMS readout. The el ~ +/-180
  samples carry ~45% of the squared power, so cutting to |el| < 30 tells you
  whether you are fitting the core or just those.
- **Gain is huge, and that is expected.** `Y` is raw correlator accumulator
  counts (per-channel data RMS ~1e5-3e7) while the model is built from the
  unit-normalized HFSS beam, so the amplitude that matches them runs ~1e7-1e11
  depending on channel and shape terms. That is a units conversion, not a
  calibration; the automated pipeline absorbs exactly the same factor (its
  fitted `gain**2` has median 1.0e11). The slider spans 1 to 1e14, and while
  auto-fit is on it parks itself on the RMS-optimal value, so switching auto-fit
  off starts you in the right decade.
- **Shape Re/Im** are the PCA shape corrections -- the model's genuine shape
  freedom beyond the HFSS prior. All zero = pure HFSS.


In [4]:

ch_w = W.Dropdown(options=[(f"ch {c}  ({f:.1f} MHz, arm {a})", i)
                           for i, (c, f, a) in enumerate(zip(CHANS, FREQS, ARMS))],
                  value=int(np.argmin(np.abs(CHANS - 712))), description="channel")
arm_w = W.Dropdown(options=[("auto (from channel)", -1), ("force arm 0", 0), ("force arm 1", 1)],
                   value=-1, description="arm")
alpha_w = W.FloatSlider(value=float(ALPHA_NEW), min=0.0, max=180.0, step=0.5,
                        description="alpha (deg)", continuous_update=False)
dE_w = W.FloatSlider(value=float(SURVEYED_DELTA[0]), min=-200.0, max=200.0, step=0.5,
                     description="TX dE (m)", continuous_update=False)
dN_w = W.FloatSlider(value=float(SURVEYED_DELTA[1]), min=-200.0, max=200.0, step=0.5,
                     description="TX dN (m)", continuous_update=False)
dU_w = W.FloatSlider(value=float(SURVEYED_DELTA[2]), min=-200.0, max=200.0, step=0.5,
                     description="TX dU (m)", continuous_update=False)
azoff_w = W.FloatSlider(value=0.0, min=-180.0, max=180.0, step=0.5,
                        description="az offset", continuous_update=False)
eloff_w = W.FloatSlider(value=0.0, min=-180.0, max=180.0, step=0.5,
                        description="el offset", continuous_update=False)
autog_w = W.Checkbox(value=True, description="auto-fit amplitude")
# Pure-HFSS baseline. The shape terms are extra, non-physical degrees of
# freedom on top of the HFSS prior; with this checked the model is the
# unperturbed physical prediction and only the physical knobs (amplitude,
# TX direction, alpha) do anything. Default ON: the baseline should be what
# you get without asking for it.
purehfss_w = W.Checkbox(value=True, description="pure HFSS (lock shape)")
# B7's per-cycle receiver gain. ON divides measured_tx by g_rx(nu, t),
# interpolated onto the sample times, and drops samples with no solution.
# Default OFF because applying it makes the fit worse -- see the Calibration
# section below. The toggle exists so both can be looked at.
cal_w = W.Checkbox(value=False, description="apply g_rx cal (B7)",
                   disabled=not E_CAL_AVAILABLE)
# Elevation rotation direction. ON = Christian's hardware convention (+EL tips
# the boresight NORTH). OFF reproduces the model as it shipped, which tips it
# SOUTH -- i.e. backwards. Default ON, but note that everything published
# before 2026-09-17 used the OFF behaviour.
elsign_w = W.Checkbox(value=True, description="correct el direction")
# Y is raw correlator accumulator counts (data RMS ~1e5-3e7) while the model is
# built from the unit-normalized HFSS beam, so the amplitude that matches them is
# ~1e5-1e12 depending on channel and shape terms (the automated pipeline's own
# fitted gain**2 has median 1.0e11). The slider spans that, with headroom.
gain_w = W.FloatLogSlider(value=1.0e9, base=10, min=0.0, max=14.0, step=0.02,
                          description="gain", continuous_update=False,
                          readout_format=".3e")
elcut_w = W.FloatSlider(value=180.0, min=5.0, max=180.0, step=5.0,
                        description="|el| cut", continuous_update=False)
shape_re = [W.FloatSlider(value=0.0, min=-2.0, max=2.0, step=0.01,
                          description=f"shape{j+1} Re", continuous_update=False)
            for j in range(NCOMP - 1)]
shape_im = [W.FloatSlider(value=0.0, min=-2.0, max=2.0, step=0.01,
                          description=f"shape{j+1} Im", continuous_update=False)
            for j in range(NCOMP - 1)]

btn_surveyed = W.Button(description="geometry: surveyed", button_style="success")
btn_fitted = W.Button(description="geometry: pipeline fit")
btn_old = W.Button(description="geometry: old consensus")
btn_zero = W.Button(description="zero the shape terms")


def _set_heading(h, alpha):
    # sliders are in metres; scale the unit heading to a representative range
    v = np.asarray(h, float) * np.linalg.norm(SURVEYED_DELTA)
    dE_w.value, dN_w.value, dU_w.value = float(v[0]), float(v[1]), float(v[2])
    alpha_w.value = float(alpha)


btn_surveyed.on_click(lambda _: _set_heading(
    SURVEYED_DELTA / np.linalg.norm(SURVEYED_DELTA), alpha_w.value))
btn_fitted.on_click(lambda _: _set_heading(HEADING_NEW, ALPHA_NEW))
btn_old.on_click(lambda _: _set_heading(HEADING_OLD, ALPHA_OLD))


def _zero_shapes(_):
    for s in shape_re + shape_im:
        s.value = 0.0


btn_zero.on_click(_zero_shapes)
print("controls built")


controls built


In [5]:

out = W.Output()
fig = None


def update(**kw):
    global fig
    i = ch_w.value
    arm = int(ARMS[i]) if arm_w.value == -1 else int(arm_w.value)
    heading = heading_from_enu(dE_w.value, dN_w.value, dU_w.value)
    gain = None if autog_w.value else gain_w.value
    use_cal = bool(cal_w.value) and E_CAL_AVAILABLE
    el_sign = -1 if elsign_w.value else 1
    locked = purehfss_w.value
    if locked:
        # hold the shape terms at zero AND grey them out, so the pure-HFSS
        # baseline cannot be perturbed by a stray drag
        for w in shape_re + shape_im:
            if w.value != 0.0:
                w.unobserve_all()
                w.value = 0.0
                w.observe(lambda change: update(), names="value")
            w.disabled = True
        s_re = [0.0] * len(shape_re)
        s_im = [0.0] * len(shape_im)
    else:
        for w in shape_re + shape_im:
            w.disabled = False
        s_re = [w.value for w in shape_re]
        s_im = [w.value for w in shape_im]
    m, A = model_power(i, heading, alpha_w.value, arm, s_re, s_im,
                       gain=gain, az_off=azoff_w.value, el_off=eloff_w.value,
                       apply_cal=use_cal, el_sign=el_sign)
    if gain is None and A > 0:
        # park the slider on the auto-fit value so turning auto-fit off starts
        # from the RMS-optimal amplitude rather than jumping orders of magnitude
        _set_gain_silently(A)
    cut = elcut_w.value
    rms = normalized_rms(i, m, el_cut=cut, apply_cal=use_cal)
    rms_all = normalized_rms(i, m, el_cut=180.0, apply_cal=use_cal)

    u = chan_used(i, use_cal) & (np.abs(EL) <= cut)
    d = chan_data(i, use_cal)
    gd, gm = grid(d, u), grid(m, u)
    gr = grid(d - m, u)
    finite = np.isfinite(gd)
    vmax = np.nanpercentile(gd[finite], 98) if finite.any() else 1.0
    vmin = np.nanpercentile(gd[finite], 2) if finite.any() else 0.0
    rmax = np.nanpercentile(np.abs(gr[np.isfinite(gr)]), 98) if np.isfinite(gr).any() else 1.0

    with out:
        out.clear_output(wait=True)
        fig, axes = plt.subplots(1, 3, figsize=(15, 4.0))
        ext = [AZ_EDGES[0], AZ_EDGES[-1], EL_EDGES[0], EL_EDGES[-1]]
        for ax, g, ttl, cmap, lo, hi in (
                (axes[0], gd, "data", "viridis", vmin, vmax),
                (axes[1], gm, "model", "viridis", vmin, vmax),
                (axes[2], gr, "data - model", "RdBu_r", -rmax, rmax)):
            im = ax.imshow(g, origin="lower", aspect="auto", extent=ext,
                           cmap=cmap, vmin=lo, vmax=hi, interpolation="nearest")
            ax.set_title(ttl); ax.set_xlabel("az [deg]"); ax.set_ylabel("el [deg]")
            plt.colorbar(im, ax=ax)
        fig.suptitle(
            f"ch {CHANS[i]} ({FREQS[i]:.2f} MHz, arm {arm})   "
            f"normalized RMS = {rms:.4f}"
            + (f"  (|el|<{cut:.0f}; all-el {rms_all:.4f})" if cut < 180 else "")
            + f"   amplitude {A:.4g}")
        fig.tight_layout()
        plt.show()
        below = np.degrees(np.arctan2(-heading[2], np.hypot(heading[0], heading[1])))
        print(f"heading unit vector [{heading[0]:+.4f}, {heading[1]:+.4f}, {heading[2]:+.4f}]"
              f"  = {below:.1f} deg below horizontal")
        mode = ("PURE HFSS (shape terms locked at zero)" if locked
                else f"shapes Re {[round(v,3) for v in s_re]} "
                     f"Im {[round(v,3) for v in s_im]}")
        print(f"alpha {alpha_w.value:.2f} deg | az off {azoff_w.value:+.1f} | "
              f"el off {eloff_w.value:+.1f} | {mode}")
        if use_cal:
            print(f"g_rx cal ON: data is counts / g_rx, "
                  f"{int(u.sum())} samples with a solution "
                  f"(median {np.nanmedian(CAL_ANCHOR_MIN[u]):.0f} min from a "
                  f"real cycle). NOTE: this makes the fit worse -- "
                  f"see beam_cal_toggle_checkpoint.")
        else:
            print("g_rx cal OFF: data is raw accumulator counts.")
        print("el direction: " + ("CORRECTED (+EL tips boresight NORTH, "
                                  "Christian's hardware convention)"
                                  if el_sign < 0 else
                                  "as-shipped (+el tips SOUTH -- backwards; "
                                  "matches everything published before "
                                  "2026-09-17)"))


ctrl = [ch_w, arm_w, alpha_w, dE_w, dN_w, dU_w, azoff_w, eloff_w,
        autog_w, gain_w, elcut_w, purehfss_w, cal_w,
        elsign_w] + shape_re + shape_im
for w in ctrl:
    w.observe(lambda change: update(), names="value")


def _set_gain_silently(value):
    lo, hi = 10.0 ** gain_w.min, 10.0 ** gain_w.max
    v = float(min(max(value, lo), hi))
    gain_w.unobserve_all()
    gain_w.value = v
    gain_w.observe(lambda change: update(), names="value")

panel = W.VBox([
    W.HBox([ch_w, arm_w, elcut_w]),
    W.HBox([alpha_w, autog_w, gain_w]),
    W.HBox([purehfss_w, cal_w, elsign_w]),
    W.HBox([dE_w, dN_w, dU_w]),
    W.HBox([azoff_w, eloff_w]),
    W.HBox(shape_re),
    W.HBox(shape_im),
    W.HBox([btn_surveyed, btn_fitted, btn_old, btn_zero]),
])
display(panel, out)
update()


Output()


## Context you may want while twiddling

- **WITHDRAWN 2026-09-17 — "the fit is nearly blind to transmitter direction"
  was wrong.** It rested on a *two-point* comparison (the pipeline's fitted
  heading vs the surveyed one, which happen to score within ~0.0005 of each
  other) and was generalised into a claim about the whole parameter space. Two
  points that happen to tie say nothing about sensitivity. Aaron caught it.

- **The real per-axis sensitivity, measured by scanning each axis with the
  amplitude *and* `alpha` refit at every point:**

  | axis | ch 712 spread | 15-channel median spread |
  |---|---|---|
  | `dN` | **0.160** | **0.156** |
  | `dE` | 0.035 | 0.102 |
  | `dU` | 0.032 | 0.030 |

  **`dN` is the informative axis** — strongest on ch 712 and fleet-wide, with a
  well-defined optimum (ch 712 prefers `dN ≈ +28 m`, rising to 0.51 by −60 m).
  **`dU` really is weak** (~0.03 either way): the model is pure far-field, so
  range is degenerate with gain by construction and only the *direction* bites.
  **`dE` is in between, and channel-dependent** — nearly flat on ch 712 but a
  0.10 spread across the fleet, so do not treat it as free either.

  The honest one-line version: the fit constrains the transmitter *bearing*
  (mostly through `dN`), barely constrains its *range*, and the old blanket
  statement conflated the two.
- **The elevation rotation direction in this model is BACKWARDS, and the
  `correct el direction` checkbox (default ON) fixes it.** Christian
  (2026-09-17): `+EL` is the right-hand rule about the highline vector pointing
  **west**, so at `el = 0` it tips the boresight toward **north**. The model
  rotates elevation about the fixed `+x = East` shaft, giving
  `boresight_ENU = (0, −sin el, cos el)` — it tips **south**. Read straight off
  the rotation matrix, so there is no ambiguity.
  - **Effect on the fitted geometry: the transmitter bearing rotates by
    180°.** Verified exact to ~1e-5: flipping `el` is identically
    `(dE, dN) → (−dE, −dN)`. (An earlier note here said it "flips `dN`" — that
    is only the `dE = 0` slice of this more general rule.)
  - **It does *not* reconcile the fit with the survey.** Best-fit heading is
    43.2° from the surveyed direction uncorrected and **43.4° corrected** —
    the flip mirrors the optimum rather than moving it toward nadir. At the
    surveyed heading itself the corrected convention is modestly better
    (RMS 0.3684 vs 0.3856). The fit still wants the transmitter ~87 m
    horizontally (38° off vertical) where the survey puts it 6.9 m (4.2°).
  - **Everything published before 2026-09-17 used the uncorrected sign** —
    `fit_beam_v2`, both review checkpoints. Turning this on changes fitted
    headings; it does not change the arm or calibration findings.

- **The arm-to-channel wiring is now known (2026-09-17, Aaron/Christian).**
  `RX1 = 76 MHz`, `RX6 = 78 MHz`; RX6 is the arm the theodolite measured. Those
  are **ch 312** and **ch 320** — adjacent comb lines, exactly one comb spacing
  apart (1.953125 MHz). With `tx_arm_for_channel = (ch//8) % 2` that gives
  **model arm 1 = RX1 (76 MHz)** and **model arm 0 = RX6 (78 MHz)**, so
  **ch 712 is arm 1 = RX1**. Two consequences:
  - It is the **first independent support** for the `(ch//8) % 2` parity rule,
    which until now was asserted without justification: two adjacent comb lines
    really are different physical arms.
  - It settles the arm-label question that fit quality *cannot* settle (the
    relabel is an exact degeneracy — see the next bullet). Do not reassign
    ch 712 to arm 0.

- **But the fit does NOT reproduce the arms' physical 90 deg separation, and
  that is a real failure.** Fitting the field angle `a` freely per channel and
  taking the circular mean per arm gives `arm 1 = 125.6 deg`, `arm 0 = 111.3
  deg` — **14 deg apart** (stable at 14–16 deg over RMS cuts of 0.45/0.50/0.60).
  The theodolite says the two arms are **exactly 90 deg** apart. Consequently
  the two arms imply azimuth-zero offsets that disagree by **76 deg**, so no
  single az-zero reconciles them and the theodolite **cannot** be used to pin
  `alpha` yet. This is the same pathology as the arm anti-correlation already
  documented (data `r ≈ −0.92`, model `r = +0.9935`) — now stated in physical
  units: the model's two arms are nearly degenerate where the hardware's are
  orthogonal.

- **Swapping the arm label is exactly a 90 deg shift in `alpha`, and nothing
  else.** The model uses `a = alpha + 90*arm` for the field direction, so
  `arm 1 @ alpha` and `arm 0 @ alpha+90` are the *same model* — verified
  identical to 6 decimal places (both 0.391375 at alpha 51 / 141). `alpha` is
  an axis, not a vector, so it is **180 deg periodic**: −38 deg ≡ 142 deg.
  Changing the arm label therefore cannot make the fit better or worse; it only
  changes which physical arm your fitted `alpha` refers to, which is a *wiring*
  question, not a fitting one.

- **Reversing the elevation rotation direction flips the sign of `dN`.**
  Verified: `RMS(el flipped, dN)` equals `RMS(el as-is, −dN)` to ~1e-5 — but
  **only when `dE = 0`**. Algebraically `el → −el` is `dN → −dN` combined with
  `az → −az`; with `dE = 0` the heading lies in the N–U plane, which is that
  symmetry's mirror plane, so the az part drops out. With `dE ≠ 0` the
  equivalence degrades (up to ~0.06 in RMS at `dE = ±30 m`). Note the fit
  quality at the optimum is *identical* either way (0.3378), so the beam data
  **cannot tell you which convention is right** — it only gives you the sign of
  `dN` conditional on the convention you assume.

- **The beam is front/back symmetric to ~1%**, so elevation offsets of 180 deg
  are close to a no-op. Likewise the model's azimuth response is 180-deg
  periodic.
- **Polarization angle mostly reweights two fixed lobes** (~86 deg apart in az)
  rather than rotating the pattern; the `argmax` of the profile can jump 85 deg
  from a ~2% reweighting, while the profile itself barely changes.
- **The two arms carry genuinely different azimuth structure** that the model
  cannot reproduce: an empirical template from a channel's own arm fits at ~0.24
  normalized RMS, the other arm's at ~0.96, the HFSS model at ~0.67. If you find
  parameters that reproduce the arm difference, that is a real result.
- Per-channel pipeline values (`gain_fitted`, shape terms) are in the cache as
  `per_channel_gain_fitted`, `per_channel_shape_re`, `per_channel_shape_im` if
  you want to load the automated fit's answer for comparison.
